# Taller 02: Poda Alfa Beta 

### Grupo: Natalia Carpintero, Paula Núñez e Isabella Arrieta.

**Objetivo:** Diseñe e implemente un algoritmo de Poda Alfa Beta que se pueda aplicar de forma genérica a un árbol presentado usando listas de Python.

## Guía rápida para entender el árbol y la poda alfa-beta

1. **¿Cómo se representa cada nivel?**
   - El árbol se representa con **listas anidadas**.
   - Cada lista corresponde a un nodo interno.
   - Cada número corresponde a una **hoja**.
   - Ejemplo: `[[[3, 5], [6, 9]], [[1, 2], [8, 4]], [[7, 10], [2, 6]]]`.
   - El primer nivel contiene 3 hijos, cada hijo contiene 2 subárboles, y cada subárbol contiene hojas.

2. **¿Quién es MAX y quién MIN en cada nivel?**
   - La raíz se define al crear el solver con `preferencia = "max"` o `preferencia = "min"`.
   - Si la raíz es `MAX`, los niveles alternan así: `MAX -> MIN -> MAX -> MIN`.
   - Si la raíz es `MIN`, los niveles alternan así: `MIN -> MAX -> MIN -> MAX`.
   - El nivel se decide por la profundidad del nodo.

3. **¿Cómo se obtiene la secuencia de jugadas?**
   - En cada nivel se guarda el índice del hijo elegido.
   - Si la mejor jugada está en el hijo 2 del nivel raíz y luego en el hijo 1 del siguiente nivel, la secuencia parcial sería `[2, 1]`.
   - La secuencia completa se arma al unir los índices elegidos desde la raíz hasta la hoja final.
   - Esa secuencia está en `secuencia_optima`.

4. **¿Cómo se identifica exactamente una rama podada?**
   - Una rama queda podada cuando ya no puede mejorar el resultado actual.
   - En `MAX`, se poda cuando `alfa >= beta`.
   - En `MIN`, se poda cuando `alfa >= beta`.
   - La rama podada se guarda con su `path`, que indica exactamente qué posición del árbol fue descartada.
   - En la traza también aparece el motivo de la poda y los valores de `alfa` y `beta` en ese momento.

5. **¿Cómo se representa un árbol de 1, 2, 3, 4 y 5 niveles?**
   - **1 nivel:** `[3, 5, 2]`
   - **2 niveles:** `[[3, 5], [6, 9]]`
   - **3 niveles:** `[[[3, 5], [6, 9]], [[1, 2], [8, 4]]]`
   - **4 niveles:** `[[[[3], [5]], [[6], [9]]], [[[1], [2]], [[8], [4]]]]`
   - **5 niveles:** `[[[[[3]], [[5]]], [[[6]], [[9]]]], [[[[1]], [[2]]], [[[8]], [[4]]]]]`

La idea general es leer el árbol de arriba hacia abajo, alternando MAX y MIN, y seguir solo la ruta que produce el mejor valor sin explorar ramas que ya no aportan.

## Descripción de la implementación

Esta notebook implementa poda alfa-beta usando una clase `AlphaBetaSolver`.

- La preferencia inicial de la raíz se define al crear el objeto con `preferencia = "max"` o `preferencia = "min"`.
- El árbol se representa con listas anidadas de Python.
- El método `minimax(...)` calcula el valor óptimo sin poda.
- El método `alpha_beta_pruning(...)` calcula el valor óptimo con poda alfa-beta y además guarda una traza paso a paso.
- El método `render_tree(...)` imprime el árbol en texto, mostrando comparaciones, actualizaciones de alfa y beta, y ramas podadas.

La estructura general sigue este flujo:

1. Se normaliza la preferencia de la raíz.
2. Se recorren los niveles alternando MAX y MIN.
3. Se actualizan `alfa` y `beta` en cada nodo.
4. Cuando se cumple el criterio de poda, se registran las ramas descartadas.
5. Al final se imprime una vista textual del árbol para revisar el proceso.

In [24]:
from math import inf


class AlphaBetaSolver:
    def __init__(self, preference="max"):
        # La preferencia de la raíz define si el primer nivel actúa como MAX o MIN.
        self.preference = self.normalize_preference(preference)

    def normalize_preference(self, preference):
        """Normaliza la preferencia de la raíz a 'max' o 'min'."""
        preference = preference.strip().lower()
        if preference not in {"max", "min"}:
            raise ValueError("La preferencia debe ser 'max' o 'min'.")
        return preference

    def is_leaf(self, node):
        """Una hoja es cualquier valor que no sea una lista."""
        return not isinstance(node, list)

    def is_max_turn(self, depth):
        """Alterna MAX y MIN según la profundidad y la preferencia de la raíz."""
        root_is_max = self.preference == "max"
        if root_is_max:
            return depth % 2 == 0
        return depth % 2 == 1

    def evaluate_tree(self, tree, depth=0):
        """Calcula el valor minimax del subárbol según la preferencia raíz.""
        Evalúa el valor final de un subárbol sin guardar la ruta."""
        if self.is_leaf(tree):
            return tree

        child_values = []
        for child in tree:
            child_value = self.evaluate_tree(child, depth + 1)
            child_values.append(child_value)

        if self.is_max_turn(depth):
            return max(child_values)
        return min(child_values)

    def minimax(self, tree, depth=0):
        """Minimax sin poda. Devuelve el valor óptimo y la secuencia elegida."""
        if self.is_leaf(tree):
            return {"value": tree, "sequence": []}

        if self.is_max_turn(depth):
            best_value = -inf
            best_sequence = []
            for i, child in enumerate(tree):
                result = self.minimax(child, depth + 1)
                candidate_value = result["value"]
                if candidate_value > best_value:
                    best_value = candidate_value
                    best_sequence = [i] + result["sequence"]
            return {"value": best_value, "sequence": best_sequence}

        best_value = inf
        best_sequence = []
        for i, child in enumerate(tree):
            result = self.minimax(child, depth + 1)
            candidate_value = result["value"]
            if candidate_value < best_value:
                best_value = candidate_value
                best_sequence = [i] + result["sequence"]
        return {"value": best_value, "sequence": best_sequence}

    def _mark_pruned_branch(self, path, reason, alpha, beta, pruned, trace, depth, node_type):
        # Guarda la rama descartada y deja evidencia en la traza para poder mostrarla después.
        pruned.append(
            {
                "depth": depth,
                "node_type": node_type,
                "path": path,
                "reason": reason,
                "alpha": alpha,
                "beta": beta,
            }
        )
        trace.append(
            {
                "event": "prune",
                "depth": depth,
                "path": path[:-1],
                "node_type": node_type,
                "pruned_path": path,
                "reason": reason,
                "alpha": alpha,
                "beta": beta,
            }
        )

    def alpha_beta(self, tree, depth=0, alpha=-inf, beta=inf, path=None, pruned=None, trace=None):
        """Aplica poda alfa-beta y registra el recorrido paso a paso."""
        if path is None:
            path = []
        if pruned is None:
            pruned = []
        if trace is None:
            trace = []

        # Identifica si este nivel del árbol se comporta como MAX o MIN.
        node_type = "MAX" if self.is_max_turn(depth) else "MIN"
        trace.append(
            {
                "event": "enter",
                "depth": depth,
                "path": path[:],
                "node_type": node_type,
                "alpha": alpha,
                "beta": beta,
            }
        )

        # Si llegamos a una hoja, simplemente devolvemos su valor.
        if self.is_leaf(tree):
            trace.append(
                {
                    "event": "leaf",
                    "depth": depth,
                    "path": path[:],
                    "node_type": "LEAF",
                    "value": tree,
                    "alpha": alpha,
                    "beta": beta,
                }
            )
            return {
                "value": tree,
                "sequence": path,
                "alpha": alpha,
                "beta": beta,
                "pruned": pruned.copy(),
                "trace": trace.copy(),
                "node_type": "LEAF",
            }

        # En MAX buscamos el valor más alto posible.
        if self.is_max_turn(depth):
            best_value = -inf
            best_sequence = path[:]
            for i, child in enumerate(tree):
                child_path = path + [i]
                trace.append(
                    {
                        "event": "compare",
                        "depth": depth,
                        "path": path[:],
                        "node_type": "MAX",
                        "branch_index": i,
                        "child_path": child_path,
                        "alpha": alpha,
                        "beta": beta,
                    }
                )
                result = self.alpha_beta(
                    child,
                    depth + 1,
                    alpha,
                    beta,
                    child_path,
                    pruned,
                    trace,
                )
                candidate_value = result["value"]
                if candidate_value > best_value:
                    best_value = candidate_value
                    best_sequence = result["sequence"]

                trace.append(
                    {
                        "event": "update",
                        "depth": depth,
                        "path": path[:],
                        "node_type": "MAX",
                        "branch_index": i,
                        "candidate": candidate_value,
                        "best_value": best_value,
                        "alpha": alpha,
                        "beta": beta,
                    }
                )

                alpha = max(alpha, best_value)
                # Si alfa alcanza o supera beta, las ramas restantes ya no aportan.
                if alpha >= beta:
                    for j in range(i + 1, len(tree)):
                        sibling_path = path + [j]
                        self._mark_pruned_branch(
                            sibling_path,
                            "MAX corta por beta",
                            alpha,
                            beta,
                            pruned,
                            trace,
                            depth,
                            "MAX",
                        )
                    break

            return {
                "value": best_value,
                "sequence": best_sequence,
                "alpha": alpha,
                "beta": beta,
                "pruned": pruned.copy(),
                "trace": trace.copy(),
                "node_type": "MAX",
            }

        # En MIN buscamos el valor más bajo posible.
        best_value = inf
        best_sequence = path[:]
        for i, child in enumerate(tree):
            child_path = path + [i]
            trace.append(
                {
                    "event": "compare",
                    "depth": depth,
                    "path": path[:],
                    "node_type": "MIN",
                    "branch_index": i,
                    "child_path": child_path,
                    "alpha": alpha,
                    "beta": beta,
                }
            )
            result = self.alpha_beta(
                child,
                depth + 1,
                alpha,
                beta,
                child_path,
                pruned,
                trace,
            )
            candidate_value = result["value"]
            if candidate_value < best_value:
                best_value = candidate_value
                best_sequence = result["sequence"]

            trace.append(
                {
                    "event": "update",
                    "depth": depth,
                    "path": path[:],
                    "node_type": "MIN",
                    "branch_index": i,
                    "candidate": candidate_value,
                    "best_value": best_value,
                    "alpha": alpha,
                    "beta": beta,
                }
            )

            beta = min(beta, best_value)
            # Si alfa alcanza o supera beta, las ramas restantes se podan.
            if alpha >= beta:
                for j in range(i + 1, len(tree)):
                    sibling_path = path + [j]
                    self._mark_pruned_branch(
                        sibling_path,
                        "MIN corta por alpha",
                        alpha,
                        beta,
                        pruned,
                        trace,
                        depth,
                        "MIN",
                    )
                break

        return {
            "value": best_value,
            "sequence": best_sequence,
            "alpha": alpha,
            "beta": beta,
            "pruned": pruned.copy(),
            "trace": trace.copy(),
            "node_type": "MIN",
        }

    def alpha_beta_pruning(self, tree):
        """Punto de entrada principal para resolver un árbol completo."""
        result = self.alpha_beta(
            tree,
            depth=0,
            alpha=-inf,
            beta=inf,
            path=[],
            pruned=[],
            trace=[],
        )
        return {
            "valor": result["value"],
            "secuencia_optima": result["sequence"],
            "alfa_final": result["alpha"],
            "beta_final": result["beta"],
            "ramas_podadas": result["pruned"],
            "traza": result["trace"],
            "preferencia_raiz": self.preference,
        }

    def _path_to_text(self, path):
        """Convierte una ruta como [2, 1, 0] en un texto legible."""
        if not path:
            return "raíz"
        return "-".join(map(str, path))

    def _trace_for_path(self, trace, path, event_name=None, branch_index=None):
        """Filtra eventos de la traza que pertenecen a una ruta concreta."""
        matches = []
        for event in trace:
            if event.get("path") != path:
                continue
            if event_name is not None and event.get("event") != event_name:
                continue
            if branch_index is not None and event.get("branch_index") != branch_index:
                continue
            matches.append(event)
        return matches

    def _is_pruned_path(self, path, pruned_paths):
        """Verifica si una ruta pertenece a una rama podada."""
        for pruned_path in pruned_paths:
            if len(path) >= len(pruned_path) and tuple(path[: len(pruned_path)]) == pruned_path:
                return True
        return False

    def render_tree(self, tree, result=None, depth=0, path=None):
        """Construye una versión textual del árbol con la traza y las podas."""
        if path is None:
            path = []

        trace = []
        selected_path = tuple()
        if result:
            trace = result.get("traza", [])
            selected_path = tuple(result.get("secuencia_optima", []))
            pruned_paths = set()
            for item in result.get("ramas_podadas", []):
                pruned_paths.add(tuple(item["path"]))
        else:
            pruned_paths = set()

        indent = "  " * depth
        path_label = self._path_to_text(path)

        if self.is_leaf(tree):
            lines = [f"{indent}{path_label}: hoja = {tree}"]
            leaf_events = self._trace_for_path(trace, path, "leaf")
            if leaf_events:
                event = leaf_events[-1]
                lines.append(
                    f"{indent}  evalúa hoja: valor={event['value']}, alfa={event['alpha']}, beta={event['beta']}"
                )
            if selected_path and tuple(path) == selected_path[: len(path)]:
                lines[0] = lines[0] + " [ruta óptima]"
            if self._is_pruned_path(tuple(path), pruned_paths):
                lines[0] = lines[0] + " [PODADO]"
            return lines

        node_type = "MAX" if self.is_max_turn(depth) else "MIN"
        node_value = self.evaluate_tree(tree, depth)
        lines = [f"{indent}{path_label}: {node_type} -> valor = {node_value}"]
        if selected_path and tuple(path) == selected_path[: len(path)]:
            lines[0] = lines[0] + " [ruta óptima]"

        enter_events = self._trace_for_path(trace, path, "enter")
        if enter_events:
            event = enter_events[-1]
            lines.append(
                f"{indent}  entra con alfa={event['alpha']} y beta={event['beta']}"
            )

        for i, child in enumerate(tree):
            child_path = path + [i]
            compare_events = self._trace_for_path(trace, path, "compare", i)
            if compare_events:
                event = compare_events[-1]
                child_label = self._path_to_text(child_path)
                lines.append(
                    f"{indent}  compara hijo {i} ({child_label}) con alfa={event['alpha']} y beta={event['beta']}"
                )

            if self._is_pruned_path(tuple(child_path), pruned_paths):
                lines.append(
                    f"{indent}  hijo {i} ({self._path_to_text(child_path)}): PODADO"
                )
                continue

            child_lines = self.render_tree(child, result, depth + 1, child_path)
            lines.extend(child_lines)

            update_events = self._trace_for_path(trace, path, "update", i)
            if update_events:
                event = update_events[-1]
                lines.append(
                    f"{indent}  actualiza tras hijo {i}: candidato={event['candidate']}, mejor={event['best_value']}, alfa={event['alpha']}, beta={event['beta']}"
                )

            prune_events = self._trace_for_path(trace, path, "prune", i)
            if prune_events:
                event = prune_events[-1]
                lines.append(
                    f"{indent}  poda en hijo {i} ({self._path_to_text(event['pruned_path'])}): {event['reason']} con alfa={event['alpha']} y beta={event['beta']}"
                )

        return lines

In [25]:
# ----------- Ejemplo de prueba -----------

# Cambia esto a "min" si quieres que la raíz juegue como MIN.
preferencia = "max"

#tree = [[3, -5, 2], [-5, -7, 4], [-9, 6, -8]]
tree= [[[3,5],[6,9]],[[1,2],[8,4]],[[7,10],[2,6]]]
#tree= [[[3,5],[6,9]],[[1,2],[0,-1]]]
solver = AlphaBetaSolver(preferencia)

minimax_result = solver.minimax(tree)
alpha_beta_result = solver.alpha_beta_pruning(tree)

print("Preferencia raíz:", solver.preference.upper())
print("Minimax:")
print(minimax_result)
print("Alpha-Beta:")
print(alpha_beta_result)
print()
print("Arbol evaluado con poda alfa-beta:")
rendered_tree = solver.render_tree(tree, alpha_beta_result)
for line in rendered_tree:
    print(line)

print()
print("Resumen:")
print("Valor final:", alpha_beta_result["valor"])
print("Secuencia óptima:", alpha_beta_result["secuencia_optima"])
print("Podas:")
for prune in alpha_beta_result["ramas_podadas"]:
    print(prune)

Preferencia raíz: MAX
Minimax:
{'value': 6, 'sequence': [2, 1, 1]}
Alpha-Beta:
{'valor': 6, 'secuencia_optima': [2, 1, 1], 'alfa_final': 6, 'beta_final': inf, 'ramas_podadas': [{'depth': 2, 'node_type': 'MAX', 'path': [0, 1, 1], 'reason': 'MAX corta por beta', 'alpha': 6, 'beta': 5}, {'depth': 1, 'node_type': 'MIN', 'path': [1, 1], 'reason': 'MIN corta por alpha', 'alpha': 5, 'beta': 2}], 'traza': [{'event': 'enter', 'depth': 0, 'path': [], 'node_type': 'MAX', 'alpha': -inf, 'beta': inf}, {'event': 'compare', 'depth': 0, 'path': [], 'node_type': 'MAX', 'branch_index': 0, 'child_path': [0], 'alpha': -inf, 'beta': inf}, {'event': 'enter', 'depth': 1, 'path': [0], 'node_type': 'MIN', 'alpha': -inf, 'beta': inf}, {'event': 'compare', 'depth': 1, 'path': [0], 'node_type': 'MIN', 'branch_index': 0, 'child_path': [0, 0], 'alpha': -inf, 'beta': inf}, {'event': 'enter', 'depth': 2, 'path': [0, 0], 'node_type': 'MAX', 'alpha': -inf, 'beta': inf}, {'event': 'compare', 'depth': 2, 'path': [0, 0], 